# 01 - DBSCAN Numba CUDA float32 + drop de core points

Este Colab isola a ideia central do professor: reduzir o custo da conexao entre core points mantendo uma fracao dos cores.


In [ ]:
# ============================================================
# Imports, GPU check and warnings
# ============================================================

import math
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

from sklearn.cluster import DBSCAN as SklearnDBSCAN
from sklearn.datasets import (
    make_blobs,
    make_moons,
    load_iris,
    load_wine,
    load_breast_cancer,
    load_digits,
)
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors

from numba import cuda
try:
    from numba.core.errors import NumbaPerformanceWarning
except Exception:
    NumbaPerformanceWarning = None

warnings.filterwarnings("ignore", category=FutureWarning)
if NumbaPerformanceWarning is not None:
    warnings.simplefilter("ignore", category=NumbaPerformanceWarning)

SEED = 42
np.random.seed(SEED)

print("CUDA available by Numba?", cuda.is_available())
if cuda.is_available():
    dev = cuda.get_current_device()
    gpu_name = dev.name.decode() if isinstance(dev.name, bytes) else dev.name
    print("GPU:", gpu_name)
else:
    print("Enable GPU in Colab: Runtime > Change runtime type > GPU")


In [ ]:
# ============================================================
# Baseline backend: cuML if available, otherwise sklearn
# ============================================================

HAS_CUML = False
cp = None
CuMLDBSCAN = None

try:
    import cupy as cp
    from cuml.cluster import DBSCAN as CuMLDBSCAN
    HAS_CUML = True
    print("cuML available. Baseline: cuML GPU.")
except Exception as exc:
    print("cuML not available. Baseline: sklearn CPU.")
    print("Reason:", repr(exc))

BASELINE_NAME = "cuML" if HAS_CUML else "sklearn_cpu"


In [ ]:
# ============================================================
# Dataset helpers: synthetic + real datasets
# ============================================================

RUN_MODE = "quick"  # options: "quick" or "presentation"

if RUN_MODE == "quick":
    N_SAMPLES_2D = 1800
    N_SAMPLES_32D = 900
elif RUN_MODE == "presentation":
    N_SAMPLES_2D = 6000
    N_SAMPLES_32D = 1800
else:
    raise ValueError("RUN_MODE must be 'quick' or 'presentation'.")

MIN_SAMPLES_DEFAULT = 8


def normalize_minmax(X):
    X = np.asarray(X, dtype=np.float32)
    mn = X.min(axis=0)
    mx = X.max(axis=0)
    denom = mx - mn
    denom[denom == 0.0] = 1.0
    return ((X - mn) / denom).astype(np.float32)


def estimate_eps(X, min_samples=8, quantile=0.90, sample_size=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=np.float32)
    if len(X) > sample_size:
        idx = rng.choice(len(X), size=sample_size, replace=False)
        Xs = X[idx]
    else:
        Xs = X
    nn = NearestNeighbors(n_neighbors=min_samples)
    nn.fit(Xs)
    dists, _ = nn.kneighbors(Xs)
    return float(np.quantile(dists[:, -1], quantile))


def make_synthetic_dataset(name, n_samples, seed=SEED):
    rng = np.random.default_rng(seed)

    if name == "dense_blobs_2d":
        X, y = make_blobs(
            n_samples=n_samples,
            centers=[(-3, -3), (-3, 3), (3, -3), (3, 3)],
            cluster_std=0.18,
            random_state=seed,
        )
    elif name == "heterogeneous_blobs_2d":
        n1 = int(0.40 * n_samples)
        n2 = int(0.35 * n_samples)
        n3 = n_samples - n1 - n2
        X1, _ = make_blobs(n_samples=n1, centers=[(-4, 0)], cluster_std=0.08, random_state=seed + 1)
        X2, _ = make_blobs(n_samples=n2, centers=[(0, 0)], cluster_std=0.22, random_state=seed + 2)
        X3, _ = make_blobs(n_samples=n3, centers=[(4, 0)], cluster_std=0.55, random_state=seed + 3)
        X = np.vstack([X1, X2, X3])
        y = np.concatenate([
            np.zeros(n1, dtype=np.int32),
            np.ones(n2, dtype=np.int32),
            np.full(n3, 2, dtype=np.int32),
        ])
    elif name == "dense_blobs_noise_2d":
        n_noise = int(0.20 * n_samples)
        n_blob = n_samples - n_noise
        X_blob, y_blob = make_blobs(
            n_samples=n_blob,
            centers=[(-3, -3), (-3, 3), (3, -3), (3, 3)],
            cluster_std=0.20,
            random_state=seed,
        )
        noise = rng.uniform(low=-5.5, high=5.5, size=(n_noise, 2)).astype(np.float32)
        X = np.vstack([X_blob, noise])
        y = np.concatenate([y_blob.astype(np.int32), np.full(n_noise, -1, dtype=np.int32)])
    elif name == "moons_2d":
        X, y = make_moons(n_samples=n_samples, noise=0.045, random_state=seed)
    elif name == "blobs_32d":
        X, y = make_blobs(
            n_samples=n_samples,
            centers=6,
            n_features=32,
            cluster_std=0.45,
            random_state=seed,
        )
    else:
        raise ValueError(f"Unknown synthetic dataset: {name}")

    return normalize_minmax(X), np.asarray(y, dtype=np.int32)


def load_real_dataset(name, max_samples=None, seed=SEED):
    if name == "real_iris_4d":
        data = load_iris()
    elif name == "real_wine_13d":
        data = load_wine()
    elif name == "real_breast_cancer_30d":
        data = load_breast_cancer()
    elif name == "real_digits_64d":
        data = load_digits()
    else:
        raise ValueError(f"Unknown real dataset: {name}")

    X = np.asarray(data.data, dtype=np.float32)
    y = np.asarray(data.target, dtype=np.int32)

    if max_samples is not None and len(X) > max_samples:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(X), size=max_samples, replace=False)
        X = X[idx]
        y = y[idx]

    return normalize_minmax(X), y


def build_datasets():
    configs = [
        ("dense_blobs_2d", "synthetic", N_SAMPLES_2D, "Dense 2D blobs", 0.90),
        ("heterogeneous_blobs_2d", "synthetic", N_SAMPLES_2D, "2D blobs with different densities", 0.90),
        ("dense_blobs_noise_2d", "synthetic", N_SAMPLES_2D, "Dense 2D blobs with outliers", 0.90),
        ("moons_2d", "synthetic", N_SAMPLES_2D, "Two moons", 0.88),
        ("blobs_32d", "synthetic", N_SAMPLES_32D, "32D blobs for quantization tests", 0.90),
        ("real_iris_4d", "real", None, "Iris real dataset, 4 features", 0.90),
        ("real_wine_13d", "real", None, "Wine real dataset, 13 features", 0.90),
        ("real_breast_cancer_30d", "real", None, "Breast Cancer Wisconsin real dataset, 30 features", 0.90),
        ("real_digits_64d", "real", 1500 if RUN_MODE == "quick" else None, "Digits real image dataset, 64 features", 0.90),
    ]

    datasets = {}
    for name, kind, n_samples, description, eps_quantile in configs:
        if kind == "synthetic":
            X, y = make_synthetic_dataset(name, n_samples=n_samples, seed=SEED)
        else:
            X, y = load_real_dataset(name, max_samples=n_samples, seed=SEED)
        eps = estimate_eps(X, min_samples=MIN_SAMPLES_DEFAULT, quantile=eps_quantile, seed=SEED)
        datasets[name] = {
            "X": np.ascontiguousarray(X.astype(np.float32)),
            "y_true": y,
            "eps": eps,
            "min_samples": MIN_SAMPLES_DEFAULT,
            "kind": kind,
            "description": description,
            "eps_quantile": eps_quantile,
        }
    return datasets


datasets = build_datasets()

df_datasets = pd.DataFrame([
    {
        "dataset": name,
        "kind": data["kind"],
        "shape": data["X"].shape,
        "eps": data["eps"],
        "description": data["description"],
    }
    for name, data in datasets.items()
])

print("RUN_MODE:", RUN_MODE)
print(df_datasets.to_string(index=False))


In [ ]:
# ============================================================
# Active dataset
# ============================================================

# Synthetic examples:
#   dense_blobs_2d
#   heterogeneous_blobs_2d
#   dense_blobs_noise_2d
#   moons_2d
#   blobs_32d
#
# Real examples:
#   real_iris_4d
#   real_wine_13d
#   real_breast_cancer_30d
#   real_digits_64d

DATASET_NAME = "heterogeneous_blobs_2d"

active = datasets[DATASET_NAME]
X = active["X"]
y_true = active["y_true"]
EPS = float(active["eps"])
MIN_SAMPLES = int(active["min_samples"])

print("Dataset:", DATASET_NAME)
print("Kind:", active["kind"])
print("Description:", active["description"])
print("Shape:", X.shape)
print("EPS:", EPS)
print("MIN_SAMPLES:", MIN_SAMPLES)


In [ ]:
# ============================================================
# Metrics and plotting helpers
# ============================================================

def count_clusters(labels):
    labels = np.asarray(labels)
    unique = set(labels.tolist())
    unique.discard(-1)
    return len(unique)


def noise_ratio(labels):
    labels = np.asarray(labels)
    return float(np.mean(labels == -1))


def relabel_consecutive(labels):
    labels = np.asarray(labels, dtype=np.int32)
    out = np.full(labels.shape, -1, dtype=np.int32)
    valid = [int(v) for v in np.unique(labels) if int(v) != -1]
    mapping = {old: new for new, old in enumerate(sorted(valid))}
    for old, new in mapping.items():
        out[labels == old] = new
    return out


def result_row(version, labels, elapsed, baseline_labels, info=None):
    info = {} if info is None else dict(info)
    row = {
        "version": version,
        "time_s": float(elapsed),
        "clusters": count_clusters(labels),
        "noise_%": 100.0 * noise_ratio(labels),
        "ARI_vs_baseline": adjusted_rand_score(baseline_labels, labels),
        "NMI_vs_baseline": normalized_mutual_info_score(baseline_labels, labels),
        "n_core_points": info.get("n_core_points", np.nan),
        "n_core_kept": info.get("n_core_kept", np.nan),
        "keep_ratio_real": info.get("keep_ratio_real", np.nan),
    }
    for key, value in info.items():
        if key not in row:
            row[key] = value
    return row


def print_table(df, title="Table"):
    print(title)
    print(df.where(pd.notna(df), "N/A").to_string(index=False))


def pca_or_xy(X, seed=SEED):
    if X.shape[1] == 2:
        return X[:, :2]
    return PCA(n_components=2, random_state=seed).fit_transform(X)


def plot_clusters_2d(X, labels, title):
    X2 = pca_or_xy(X)
    plt.figure(figsize=(6, 5))
    plt.scatter(X2[:, 0], X2[:, 1], c=labels, s=8, cmap="tab20", alpha=0.85, linewidths=0)
    plt.title(title)
    plt.xlabel("x1/PCA1")
    plt.ylabel("x2/PCA2")
    plt.tight_layout()
    plt.show()


In [ ]:
# ============================================================
# Baseline DBSCAN function
# ============================================================

def _to_numpy(values):
    if values is None:
        return None
    if hasattr(values, "get"):
        return values.get()
    if hasattr(values, "to_numpy"):
        return values.to_numpy()
    if cp is not None:
        try:
            return cp.asnumpy(values)
        except Exception:
            pass
    return np.asarray(values)


def run_baseline_dbscan(X, eps, min_samples):
    X32 = np.ascontiguousarray(X.astype(np.float32))
    start = time.perf_counter()

    if HAS_CUML:
        X_gpu = cp.asarray(X32, dtype=cp.float32)
        model = CuMLDBSCAN(eps=float(eps), min_samples=int(min_samples))
        labels_raw = model.fit_predict(X_gpu)
        cp.cuda.Stream.null.synchronize()
        labels = _to_numpy(labels_raw).astype(np.int32)
        backend = "cuML"
    else:
        model = SklearnDBSCAN(eps=float(eps), min_samples=int(min_samples), n_jobs=-1)
        labels = model.fit_predict(X32).astype(np.int32)
        backend = "sklearn_cpu"

    elapsed = time.perf_counter() - start
    labels = relabel_consecutive(labels)

    core_indices = getattr(model, "core_sample_indices_", None)
    if core_indices is None:
        n_core = np.nan
    else:
        try:
            n_core = int(len(_to_numpy(core_indices)))
        except Exception:
            n_core = np.nan

    info = {
        "backend": backend,
        "n_core_points": n_core,
        "n_core_kept": n_core,
        "keep_ratio_real": 1.0 if not pd.isna(n_core) else np.nan,
    }
    return labels, elapsed, backend, info


In [ ]:
# ============================================================
# Numba CUDA float32 DBSCAN with core-point drop
# ============================================================

TPB = 128


@cuda.jit
def count_core_float32_kernel(X, eps2, min_samples, core_flags, neighbor_counts):
    i = cuda.grid(1)
    n = X.shape[0]
    d = X.shape[1]
    if i >= n:
        return

    count = 0
    for j in range(n):
        dist2 = 0.0
        for k in range(d):
            diff = X[i, k] - X[j, k]
            dist2 += diff * diff
            if dist2 > eps2:
                break
        if dist2 <= eps2:
            count += 1
            if count >= min_samples:
                break
    neighbor_counts[i] = count
    core_flags[i] = 1 if count >= min_samples else 0


@cuda.jit
def init_labels_kernel(core_flags, keep_mask, labels):
    i = cuda.grid(1)
    if i >= labels.shape[0]:
        return
    labels[i] = i if core_flags[i] == 1 and keep_mask[i] == 1 else -1


@cuda.jit
def propagate_core_labels_float32_kernel(X, eps2, keep_mask, labels, changed):
    i = cuda.grid(1)
    n = X.shape[0]
    d = X.shape[1]
    if i >= n:
        return
    if keep_mask[i] == 0 or labels[i] < 0:
        return

    current_label = labels[i]
    best_label = current_label
    for j in range(n):
        if keep_mask[j] == 0 or labels[j] < 0:
            continue
        dist2 = 0.0
        for k in range(d):
            diff = X[i, k] - X[j, k]
            dist2 += diff * diff
            if dist2 > eps2:
                break
        if dist2 <= eps2:
            lj = labels[j]
            if lj >= 0 and lj < best_label:
                best_label = lj

    if best_label < current_label:
        labels[i] = best_label
        changed[0] = 1


@cuda.jit
def assign_points_to_nearest_core_float32_kernel(X, eps2, keep_mask, labels):
    i = cuda.grid(1)
    n = X.shape[0]
    d = X.shape[1]
    if i >= n:
        return
    if labels[i] >= 0:
        return

    best_label = -1
    best_dist2 = eps2
    found = 0
    for j in range(n):
        if keep_mask[j] == 0 or labels[j] < 0:
            continue
        dist2 = 0.0
        for k in range(d):
            diff = X[i, k] - X[j, k]
            dist2 += diff * diff
            if dist2 > eps2:
                break
        if dist2 <= eps2 and (found == 0 or dist2 < best_dist2):
            found = 1
            best_dist2 = dist2
            best_label = labels[j]
    labels[i] = best_label


def make_core_keep_mask(core_flags, keep_ratio=1.0, seed=SEED):
    core_flags = np.asarray(core_flags, dtype=np.int32)
    keep_mask = np.zeros_like(core_flags, dtype=np.int32)
    core_idx = np.where(core_flags == 1)[0]

    if core_idx.size == 0:
        return keep_mask
    if keep_ratio >= 1.0:
        keep_mask[core_idx] = 1
        return keep_mask

    rng = np.random.default_rng(seed)
    n_keep = max(1, int(round(core_idx.size * float(keep_ratio))))
    selected = rng.permutation(core_idx)[:n_keep]
    keep_mask[selected] = 1
    return keep_mask


def dbscan_numba_float32(X, eps, min_samples, keep_ratio=1.0, max_iter=32, seed=SEED):
    if not cuda.is_available():
        raise RuntimeError("CUDA not available by Numba.")

    start = time.perf_counter()
    X32 = np.ascontiguousarray(X.astype(np.float32))
    n = X32.shape[0]
    blocks = max(1, (n + TPB - 1) // TPB)
    eps2 = np.float32(float(eps) * float(eps))

    d_X = cuda.to_device(X32)
    d_core_flags = cuda.device_array(n, dtype=np.int32)
    d_neighbor_counts = cuda.device_array(n, dtype=np.int32)
    count_core_float32_kernel[blocks, TPB](d_X, eps2, int(min_samples), d_core_flags, d_neighbor_counts)
    cuda.synchronize()

    core_flags = d_core_flags.copy_to_host()
    keep_mask = make_core_keep_mask(core_flags, keep_ratio=keep_ratio, seed=seed)

    d_keep_mask = cuda.to_device(keep_mask)
    d_labels = cuda.device_array(n, dtype=np.int32)
    init_labels_kernel[blocks, TPB](d_core_flags, d_keep_mask, d_labels)
    cuda.synchronize()

    iterations = 0
    for _ in range(int(max_iter)):
        iterations += 1
        d_changed = cuda.to_device(np.zeros(1, dtype=np.int32))
        propagate_core_labels_float32_kernel[blocks, TPB](d_X, eps2, d_keep_mask, d_labels, d_changed)
        cuda.synchronize()
        if d_changed.copy_to_host()[0] == 0:
            break

    assign_points_to_nearest_core_float32_kernel[blocks, TPB](d_X, eps2, d_keep_mask, d_labels)
    cuda.synchronize()

    labels = relabel_consecutive(d_labels.copy_to_host())
    elapsed = time.perf_counter() - start

    n_core = int(core_flags.sum())
    n_kept = int(keep_mask.sum())
    info = {
        "n_core_points": n_core,
        "n_core_kept": n_kept,
        "keep_ratio_real": float(n_kept / max(n_core, 1)),
        "iterations": iterations,
        "keep_ratio_target": float(keep_ratio),
        "version_family": "float32_drop",
    }
    return labels, elapsed, info


In [ ]:
# ============================================================
# Run float32/drop experiments
# ============================================================

MAX_PROPAGATION_ITERS = 32

labels_baseline, time_baseline, backend, info_base = run_baseline_dbscan(X, EPS, MIN_SAMPLES)
results = [result_row(f"baseline_{backend}", labels_baseline, time_baseline, labels_baseline, info_base)]
all_labels = {f"baseline_{backend}": labels_baseline}

experiments = [
    ("numba_float32_keep_100", 1.00),
    ("drop_core_keep_75", 0.75),
    ("drop_core_keep_50", 0.50),
    ("drop_core_keep_25", 0.25),
]

for name, keep_ratio in experiments:
    labels, elapsed, info = dbscan_numba_float32(
        X,
        EPS,
        MIN_SAMPLES,
        keep_ratio=keep_ratio,
        max_iter=MAX_PROPAGATION_ITERS,
        seed=SEED,
    )
    results.append(result_row(name, labels, elapsed, labels_baseline, info))
    all_labels[name] = labels
    print(f"{name}: {elapsed:.4f}s")

df_results = pd.DataFrame(results)
df_results["speedup_vs_baseline"] = float(df_results.loc[0, "time_s"]) / df_results["time_s"]
print_table(df_results.round(4), "Float32/drop results")


In [ ]:
# ============================================================
# Plots
# ============================================================

plt.figure(figsize=(10, 4))
plt.bar(df_results["version"], df_results["time_s"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("Time (s)")
plt.title("Float32/drop time")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.bar(df_results["version"], df_results["ARI_vs_baseline"])
plt.xticks(rotation=60, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("ARI vs baseline")
plt.title("Quality loss when dropping core points")
plt.tight_layout()
plt.show()

for name, labels in all_labels.items():
    plot_clusters_2d(X, labels, f"{name} - {DATASET_NAME}")


Interpretacao: em datasets densos, muitos pontos viram core. Ao reduzir os cores mantidos, o custo de conexao tende a cair, mas ARI/NMI e numero de clusters podem piorar.
